### Defining functions

In [22]:
import pandas as pd
import requests
from sklearn.metrics import accuracy_score, f1_score, classification_report
import time

# Configuration
MODEL_URL = "http://194.171.191.228:30080/api/chat/completions"
MODEL_NAME = "llama3.2:3b"
API_TOKEN = "sk-093ed5de00ba475fb043f7cc915cf60c"

# Emotion list
EMOTIONS = ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral']


def query_llm(sentence, prompt_template, temperature=0.2):
    """Send a sentence to the LLM and get emotion prediction"""
    prompt = prompt_template.format(sentence=sentence)
    
    headers = {
        "Authorization": f"Bearer {API_TOKEN}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": 20
    }
    
    try:
        response = requests.post(MODEL_URL, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content'].strip().lower()
    except Exception as e:
        print(f"Error: {e}")
        return "error"


def extract_emotion(response):
    """Extract emotion from LLM response"""
    for emotion in EMOTIONS:
        if emotion in response:
            return emotion
    return 'neutral'


def test_prompt(sentences, true_labels, prompt_template, prompt_name, temperature=0.2):
    """Test a prompt and return results"""
    print(f"\n{'='*60}")
    print(f"Testing: {prompt_name}")
    print(f"Temperature: {temperature}")
    print(f"{'='*60}")
    
    predictions = []
    
    for i, sentence in enumerate(sentences):
        response = query_llm(sentence, prompt_template, temperature)
        emotion = extract_emotion(response)
        predictions.append(emotion)
        
        if (i + 1) % 25 == 0:
            print(f"Processed {i+1}/{len(sentences)}")
        
        time.sleep(0.1)
    
    # Calculate metrics
    acc = accuracy_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions, average='macro', zero_division=0)
    
    print(f"\nAccuracy: {acc:.3f}")
    print(f"F1-Score: {f1:.3f}")
    print(f"\nClassification Report:")
    print(classification_report(true_labels, predictions, zero_division=0))
    
    return {
        'name': prompt_name,
        'accuracy': acc,
        'f1': f1,
        'predictions': predictions,
        'temperature': temperature
    }


print("Setup complete! Functions loaded.")


Setup complete! Functions loaded.


### Loading data

In [61]:
df = pd.read_excel('combined_peer_reviewed_dataset.xlsx')  # Change to .xlsx

# CLEAN THE LABELS!
df['Corrected_emotion'] = df['Corrected_emotion'].str.lower().str.strip()

# Fix typos
df['Corrected_emotion'] = df['Corrected_emotion'].replace({
    'suprise': 'surprise',
    'surprised': 'surprise',
    'digust': 'disgust',
    'anger/surprise': 'anger',  # Pick one or exclude
})

# Check it's clean now
print("After cleaning:")
print(df['Corrected_emotion'].value_counts())
print(f"\nUnique emotions: {df['Corrected_emotion'].unique()}")

# Verify only 7 emotions
expected = {'happiness', 'sadness', 'anger', 'surprise', 'fear', 'disgust', 'neutral'}
actual = set(df['Corrected_emotion'].unique())
unexpected = actual - expected

if unexpected:
    print(f"\n⚠ WARNING: Unexpected labels found: {unexpected}")
else:
    print(f"\n✓ All labels are valid!")

df_sample = df.head(200)  # Start with 200 for testing

sentences = df_sample['Sentence'].tolist()
true_labels = df_sample['Corrected_emotion'].tolist()  # Already lowercase now

print(f"\n✓ Loaded {len(df_sample)} sentences")

all_results = []

After cleaning:
Corrected_emotion
neutral      1818
happiness     628
sadness       343
surprise      273
anger         262
fear          208
disgust       102
Name: count, dtype: int64

Unique emotions: ['neutral' 'sadness' 'happiness' 'fear' 'anger' 'surprise' 'disgust' nan]

⚠ WARNING: Unexpected labels found: {nan}

✓ Loaded 200 sentences


### experiment 1

In [50]:
prompt1 = """Classify this sentence as one emotion: happiness, sadness, anger, surprise, fear, disgust, or neutral.

Sentence: {sentence}

Emotion:"""

result1 = test_prompt(sentences, true_labels, prompt1, "Baseline", temperature=0.2)
all_results.append(result1)


Testing: Baseline
Temperature: 0.2
Processed 25/200
Processed 50/200
Processed 75/200
Processed 100/200
Processed 125/200
Processed 150/200
Processed 175/200
Processed 200/200

Accuracy: 0.555
F1-Score: 0.420

Classification Report:
              precision    recall  f1-score   support

       anger       0.75      0.56      0.64        16
     disgust       0.60      0.43      0.50         7
        fear       1.00      0.11      0.20        18
   happiness       0.52      0.65      0.58        43
     neutral       0.53      0.75      0.62        76
     sadness       0.71      0.53      0.61        19
    surprise       0.33      0.15      0.21        13
   surprise        0.00      0.00      0.00         8

    accuracy                           0.56       200
   macro avg       0.56      0.40      0.42       200
weighted avg       0.57      0.56      0.52       200



### experiment 2

In [51]:
prompt2 = """Classify the emotion in this sentence:

- happiness: joy, satisfaction, pleasure
- sadness: sorrow, disappointment, grief  
- anger: frustration, irritation, rage
- surprise: astonishment, unexpectedness
- fear: anxiety, worry, threat
- disgust: revulsion, distaste
- neutral: no clear emotion

Sentence: {sentence}

Emotion (one word):"""

result2 = test_prompt(sentences, true_labels, prompt2, "With Definitions", temperature=0.2)
all_results.append(result2)



Testing: With Definitions
Temperature: 0.2
Processed 25/200
Processed 50/200
Processed 75/200
Processed 100/200
Processed 125/200
Processed 150/200
Processed 175/200
Processed 200/200

Accuracy: 0.490
F1-Score: 0.297

Classification Report:
              precision    recall  f1-score   support

       anger       0.50      0.06      0.11        16
     disgust       0.33      0.29      0.31         7
        fear       0.67      0.11      0.19        18
   happiness       1.00      0.28      0.44        43
     neutral       0.45      0.95      0.61        76
     sadness       0.75      0.32      0.44        19
    surprise       0.33      0.23      0.27        13
   surprise        0.00      0.00      0.00         8

    accuracy                           0.49       200
   macro avg       0.50      0.28      0.30       200
weighted avg       0.59      0.49      0.42       200



### experiment 3

In [52]:
prompt3 = """Classificeer de emotie in Nederlandse zinnen.

Kies uit: happiness, sadness, anger, surprise, fear, disgust, neutral

Voorbeelden:
"Ik ben zo blij dat we gewonnen hebben!" → happiness
"Dit is echt verschrikkelijk en teleurstellend." → sadness
"Dat maakt me zo boos!" → anger
"Wow, dat had ik nooit verwacht!" → surprise
"Ik ben bang dat dit fout gaat." → fear
"Dat vind ik echt walgelijk." → disgust
"We gaan naar de volgende opdracht." → neutral

Zin: {sentence}

Emotie (één woord):"""

result3 = test_prompt(sentences, true_labels, prompt3, "Few-Shot Dutch", temperature=0.2)
all_results.append(result3)


Testing: Few-Shot Dutch
Temperature: 0.2
Processed 25/200
Processed 50/200
Processed 75/200
Processed 100/200
Processed 125/200
Processed 150/200
Processed 175/200
Processed 200/200

Accuracy: 0.620
F1-Score: 0.478

Classification Report:
              precision    recall  f1-score   support

       anger       0.67      0.38      0.48        16
     disgust       0.62      0.71      0.67         7
        fear       0.62      0.28      0.38        18
   happiness       0.86      0.56      0.68        43
     neutral       0.58      0.92      0.71        76
     sadness       0.62      0.53      0.57        19
    surprise       0.36      0.31      0.33        13
   surprise        0.00      0.00      0.00         8

    accuracy                           0.62       200
   macro avg       0.54      0.46      0.48       200
weighted avg       0.62      0.62      0.59       200



### experiment 4

In [53]:

prompt4 = """Je bent een emotie-classificatie expert voor Nederlandse tekst.

TAAK: Classificeer de emotie in exact één woord.

EMOTIES:
- happiness: vreugde, blijdschap, tevredenheid
- sadness: verdriet, teleurstelling, somberheid
- anger: boosheid, irritatie, frustratie
- surprise: verbazing, schok
- fear: angst, bezorgdheid, spanning
- disgust: walging, afkeer
- neutral: geen emotie, feitelijk

BELANGRIJKE REGELS:
- Kies de DOMINANTE emotie, ook als deze subtiel is
- Kies NIET te snel voor neutral
- Let op context en toon

ZIN: {sentence}

ANTWOORD (één woord):"""

result4 = test_prompt(sentences, true_labels, prompt4, "Explicit Instructions", temperature=0.2)
all_results.append(result4)


Testing: Explicit Instructions
Temperature: 0.2
Processed 25/200
Processed 50/200
Processed 75/200
Processed 100/200
Processed 125/200
Processed 150/200
Error: HTTPConnectionPool(host='194.171.191.228', port=30080): Read timed out. (read timeout=30)
Processed 175/200
Processed 200/200

Accuracy: 0.380
F1-Score: 0.069

Classification Report:
              precision    recall  f1-score   support

       anger       0.00      0.00      0.00        16
     disgust       0.00      0.00      0.00         7
        fear       0.00      0.00      0.00        18
   happiness       0.00      0.00      0.00        43
     neutral       0.38      1.00      0.55        76
     sadness       0.00      0.00      0.00        19
    surprise       0.00      0.00      0.00        13
   surprise        0.00      0.00      0.00         8

    accuracy                           0.38       200
   macro avg       0.05      0.12      0.07       200
weighted avg       0.14      0.38      0.21       200



### experiment 5

In [54]:
prompt5 = """Analyseer de emotie in deze "Wie is de Mol" dialoog.

Context: Reality TV competitie met spanning, frustratie, en vreugde.

Classificeer als: happiness, sadness, anger, surprise, fear, disgust, neutral

Voorbeelden uit de show:
"Geweldig gedaan, team!" → happiness
"Ik baal hier echt van." → sadness
"Dit irriteert me mateloos!" → anger
"Wat?! Ik had dat niet verwacht!" → surprise
"Ik ben bang dat we verliezen." → fear
"Dat gedrag is walgelijk." → disgust
"We beginnen aan de opdracht." → neutral

Analyseer:
{sentence}

Emotie:"""

result5 = test_prompt(sentences, true_labels, prompt5, "TV Context", temperature=0.2)
all_results.append(result5)



Testing: TV Context
Temperature: 0.2
Processed 25/200
Processed 50/200
Processed 75/200
Processed 100/200
Processed 125/200
Processed 150/200
Processed 175/200
Processed 200/200

Accuracy: 0.520
F1-Score: 0.402

Classification Report:
              precision    recall  f1-score   support

       anger       0.50      0.56      0.53        16
     disgust       0.44      0.57      0.50         7
        fear       0.67      0.22      0.33        18
   happiness       0.58      0.51      0.54        43
     neutral       0.55      0.71      0.62        76
     sadness       0.47      0.37      0.41        19
    surprise       0.25      0.31      0.28        13
   surprise        0.00      0.00      0.00         8

    accuracy                           0.52       200
   macro avg       0.43      0.41      0.40       200
weighted avg       0.51      0.52      0.50       200



### comparing results

In [55]:
print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)

results_df = pd.DataFrame([
    {
        'Prompt': r['name'], 
        'F1-Score': f"{r['f1']:.3f}",
        'Accuracy': f"{r['accuracy']:.3f}",
        'Temperature': r['temperature']
    }
    for r in all_results
])

print(results_df.to_string(index=False))

# Best prompt
best = max(all_results, key=lambda x: x['f1'])
print(f"\nBEST PROMPT: {best['name']}")
print(f"   F1-Score: {best['f1']:.3f}")
print(f"   Accuracy: {best['accuracy']:.3f}")




RESULTS SUMMARY
               Prompt F1-Score Accuracy  Temperature
             Baseline    0.420    0.555          0.2
     With Definitions    0.297    0.490          0.2
       Few-Shot Dutch    0.478    0.620          0.2
Explicit Instructions    0.069    0.380          0.2
           TV Context    0.402    0.520          0.2

BEST PROMPT: Few-Shot Dutch
   F1-Score: 0.478
   Accuracy: 0.620


### things to try:

Use more data
try bigger model
add on to best performing prompt or try mixing techniqques

### Trying more data on best performing prompt with bigger model

In [56]:
# Switch to the bigger model
MODEL_NAME = "meta-llama/Llama-3.3-70B-Instruct"

prompt_best = """Classificeer de emotie in Nederlandse zinnen.

Kies uit: happiness, sadness, anger, surprise, fear, disgust, neutral

Voorbeelden:
"Ik ben zo blij dat we gewonnen hebben!" → happiness
"Dit is echt verschrikkelijk en teleurstellend." → sadness
"Dat maakt me zo boos!" → anger
"Wow, dat had ik nooit verwacht!" → surprise
"Ik ben bang dat dit fout gaat." → fear
"Dat vind ik echt walgelijk." → disgust
"We gaan naar de volgende opdracht." → neutral

Zin: {sentence}

Emotie (één woord):"""

print(" Testing 70B model on all 1050 sentences...")

result_70b_all = test_prompt(sentences, true_labels, prompt_best,
                             "Few-Shot Dutch 70B (All Data)", temperature=0.2)
all_results.append(result_70b_all)

if result_70b_all['f1'] >= 0.85:
    print("\ TARGET ACHIEVED! ")
else:
    print(f"\n Current: {result_70b_all['f1']:.3f}")
    print(f" Need: {0.85 - result_70b_all['f1']:.3f} more")

 Testing 70B model on all 1050 sentences...

Testing: Few-Shot Dutch 70B (All Data)
Temperature: 0.2


<>:28: SyntaxWarning: invalid escape sequence '\ '
<>:28: SyntaxWarning: invalid escape sequence '\ '
C:\Users\Flori\AppData\Local\Temp\ipykernel_26900\1681139096.py:28: SyntaxWarning: invalid escape sequence '\ '
  print("\ TARGET ACHIEVED! ")


Processed 25/200
Processed 50/200
Processed 75/200
Processed 100/200
Processed 125/200
Processed 150/200
Processed 175/200
Processed 200/200

Accuracy: 0.620
F1-Score: 0.482

Classification Report:
              precision    recall  f1-score   support

       anger       0.67      0.38      0.48        16
     disgust       0.71      0.71      0.71         7
        fear       0.62      0.28      0.38        18
   happiness       0.86      0.56      0.68        43
     neutral       0.58      0.92      0.71        76
     sadness       0.59      0.53      0.56        19
    surprise       0.36      0.31      0.33        13
   surprise        0.00      0.00      0.00         8

    accuracy                           0.62       200
   macro avg       0.55      0.46      0.48       200
weighted avg       0.62      0.62      0.59       200


 Current: 0.482
 Need: 0.368 more


### trying to improve upon the prompt

In [62]:
prompt_enhanced = """Classificeer de emotie in Nederlandse zinnen. Dit is belangrijk: kies NIET te snel voor neutral - de meeste zinnen hebben een emotie!

Kies uit: happiness, sadness, anger, surprise, fear, disgust, neutral

Voorbeelden:

HAPPINESS (blijdschap, vreugde):
"Ik ben zo blij dat we gewonnen hebben!" → happiness
"Wat geweldig, dit is fantastisch!" → happiness

SADNESS (verdriet, teleurstelling):
"Dit is echt verschrikkelijk en teleurstellend." → sadness
"Ik baal hier zo van, wat jammer." → sadness

ANGER (boosheid, irritatie):
"Dat maakt me zo boos!" → anger
"Dit irriteert me echt mateloos!" → anger

SURPRISE (verbazing):
"Wow, dat had ik nooit verwacht!" → surprise
"Wat?! Echt waar?" → surprise

FEAR (angst, bezorgdheid):
"Ik ben bang dat dit fout gaat." → fear
"Dit maakt me nerveus." → fear

DISGUST (walging):
"Dat vind ik echt walgelijk." → disgust
"Dat gedrag is afschuwelijk." → disgust

NEUTRAL (geen emotie):
"We gaan naar de volgende opdracht." → neutral
"De opdracht begint nu." → neutral

BELANGRIJK: Let op subtiele emoties. Zelfs lichte irritatie = anger, lichte bezorgdheid = fear.

Zin: {sentence}

Emotie (één woord):"""

result_enhanced = test_prompt(sentences, true_labels, prompt_enhanced,
                              "Enhanced Few-Shot 70B", temperature=0.2)
all_results.append(result_enhanced)

print(f"\n Improvement over previous:")
print(f"   Previous: 0.552")
print(f"   Current:  {result_enhanced['f1']:.3f}")
print(f"   Change:   {result_enhanced['f1'] - 0.552:+.3f}")


Testing: Enhanced Few-Shot 70B
Temperature: 0.2
Processed 25/200
Processed 50/200
Processed 75/200
Processed 100/200
Processed 125/200
Processed 150/200
Processed 175/200
Processed 200/200

Accuracy: 0.595
F1-Score: 0.536

Classification Report:
              precision    recall  f1-score   support

       anger       0.43      0.62      0.51        16
     disgust       0.44      0.57      0.50         7
        fear       0.55      0.33      0.41        18
   happiness       0.70      0.74      0.72        43
     neutral       0.64      0.64      0.64        76
     sadness       0.63      0.63      0.63        19
    surprise       0.40      0.29      0.33        21

    accuracy                           0.59       200
   macro avg       0.54      0.55      0.54       200
weighted avg       0.59      0.59      0.59       200


 Improvement over previous:
   Previous: 0.552
   Current:  0.536
   Change:   -0.016


### trying chain of thought

In [58]:


prompt_cot = """Analyseer deze Nederlandse zin stap voor stap:

Zin: {sentence}

Stap 1: Bevat de zin emotionele woorden? (blij, boos, bang, etc.)
Stap 2: Wat is de toon? (positief, negatief, neutraal, verrast)
Stap 3: Is er spanning, frustratie, teleurstelling, of vreugde?
Stap 4: Kies de dominante emotie

Emoties: happiness, sadness, anger, surprise, fear, disgust, neutral

LET OP: Kies alleen neutral als er echt geen emotie is.

Voorbeelden:
"Dat irriteert me!" → Stap 1: "irriteert" = emotie | Stap 2: negatief | Stap 3: frustratie | EMOTIE: anger
"Wow, echt?" → Stap 1: "Wow" = emotie | Stap 2: verrast | Stap 3: verbazing | EMOTIE: surprise

Geef alleen de finale emotie (één woord):"""

result_cot = test_prompt(sentences, true_labels, prompt_cot,
                        "Chain-of-Thought 70B", temperature=0.2)
all_results.append(result_cot)


Testing: Chain-of-Thought 70B
Temperature: 0.2
Processed 25/200
Processed 50/200
Processed 75/200
Processed 100/200
Processed 125/200
Processed 150/200
Processed 175/200
Processed 200/200

Accuracy: 0.620
F1-Score: 0.455

Classification Report:
              precision    recall  f1-score   support

       anger       0.75      0.56      0.64        16
     disgust       0.50      0.29      0.36         7
        fear       0.80      0.22      0.35        18
   happiness       0.69      0.72      0.70        43
     neutral       0.63      0.80      0.71        76
     sadness       0.56      0.74      0.64        19
    surprise       0.25      0.23      0.24        13
   surprise        0.00      0.00      0.00         8

    accuracy                           0.62       200
   macro avg       0.52      0.45      0.46       200
weighted avg       0.61      0.62      0.59       200



In [59]:
print(df['Corrected_emotion'].value_counts())
# Are there variants like "Surprise", "surprise", etc.?

Corrected_emotion
neutral           1814
happiness          628
sadness            341
anger              259
surprise           255
fear               208
disgust             99
surprise            12
surprised            3
neutral              3
disgust              2
suprise              2
sadness              2
anger                2
suprise              1
anger/surprise       1
digust               1
 neutral             1
Name: count, dtype: int64


In [ ]:

prompt_ultimate = """Je bent een expert in emotie-analyse voor Nederlandse TV-dialogen. Je taak is om PRECIES de dominante emotie te identificeren.


INSTRUCTIES:

1. Lees de zin zorgvuldig
2. Identificeer emotionele signalen (woorden, toon, context)
3. Kies de STERKSTE emotie - zelfs als subtiel
4. Gebruik ALLEEN deze labels: happiness, sadness, anger, surprise, fear, disgust, neutral
5. Antwoord met EXACT één woord

BELANGRIJK: Kies alleen 'neutral' als er ECHT geen emotie is. Bij twijfel: kies de emotie!


EMOTIE DEFINITIES & VOORBEELDEN:


HAPPINESS= Blijdschap, vreugde, enthousiasme, opluchting, tevredenheid
"Wat geweldig, we hebben gewonnen!" → happiness
 Gelukkig is het goed afgelopen!" → happiness

SADNESS= Verdriet, teleurstelling, spijt, somberheid, rouw
"Ik baal hier zo verschrikkelijk van." → sadness
"Wat jammer dat het niet gelukt is." → sadness

ANGER= Boosheid, irritatie, frustratie, woede, ergernis
 "Dit irriteert me echt mateloos!" → anger
 "Wat een onzin, dit slaat nergens op!" → anger
[LET OP: Ook lichte irritatie of frustratie = anger!]

SURPRISE= Verbazing, schok, onverwachtheid, verstomming
"Wat?! Dat had ik nooit verwacht!" → surprise
"Oh wow, echt waar? Ongelofelijk!" → surprise
[LET OP: "Wat?", "Echt?", "Oh!" zijn vaak surprise]

FEAR= Angst, bezorgdheid, nervositeit, spanning, onzekerheid
 "Ik ben bang dat we dit verliezen." → fear
 "Dit maakt me echt nerveus." → fear
[LET OP: Ook bezorgdheid of twijfel = fear]

DISGUST= Walging, afkeer, minachting, weerzin
 "Dat vind ik echt walgelijk gedrag." → disgust
 "Wat een smerig iets!" → disgust

NEUTRAL= Geen emotie - alleen feiten, observaties, instructies
"We beginnen nu met de opdracht." → neutral
 "De groep bestaat uit tien mensen." → neutral
[Alleen voor PURE feitelijke uitspraken zonder emotionele lading]


VEELVOORKOMENDE FOUTEN (VERMIJD DIT):

✗ "Dat is lastig" → NIET neutral, maar sadness of fear
✗ "Nou ja..." → NIET neutral, maar sadness of anger
✗ "Hmm, oké dan" → NIET neutral, maar sadness of resignation
✗ Vragende toon zonder vraagwoord → KAN surprise zijn


NU CLASSIFICEREN:


ZIN: {sentence}

CLASSIFICATIE (één enkel woord):"""

# Test it
result_ultimate = test_prompt(sentences, true_labels, prompt_ultimate,
                              "Ultimate Combined Prompt 70B", 
                              temperature=0.15)  # Slightly lower temp for consistency

all_results.append(result_ultimate)

print(f"\n{'='*60}")
print(f" COMPARISON WITH PREVIOUS BEST")
print(f"{'='*60}")
print(f"Previous best:  F1 = 0.536")
print(f"Ultimate prompt: F1 = {result_ultimate['f1']:.3f}")
print(f"Change:         {result_ultimate['f1'] - 0.536:+.3f}")

if result_ultimate['f1'] > 0.536:
    print("\n IMPROVEMENT!")
else:
    print("\nPrevious prompt remains best")


Testing: Ultimate Combined Prompt 70B
Temperature: 0.15
Processed 25/200
Processed 50/200
Processed 75/200
Processed 100/200
Processed 125/200
Processed 150/200
Processed 175/200
Processed 200/200

Accuracy: 0.620
F1-Score: 0.581

Classification Report:
              precision    recall  f1-score   support

       anger       0.46      0.69      0.55        16
     disgust       0.67      0.57      0.62         7
        fear       0.60      0.33      0.43        18
   happiness       0.73      0.70      0.71        43
     neutral       0.65      0.67      0.66        76
     sadness       0.57      0.63      0.60        19
    surprise       0.53      0.48      0.50        21

    accuracy                           0.62       200
   macro avg       0.60      0.58      0.58       200
weighted avg       0.63      0.62      0.62       200


📊 COMPARISON WITH PREVIOUS BEST
Previous best:  F1 = 0.536
Ultimate prompt: F1 = 0.581
Change:         +0.045

🎉 IMPROVEMENT!


In [ ]:
print("="*60)
print(" FINAL TEST: ALL DATA")
print("="*60)

# Load ALL data
df_sample = df  # Everything!

sentences = df_sample['Sentence'].tolist()
true_labels = df_sample['Corrected_emotion'].tolist()

print(f"\nTesting on {len(df_sample)} sentences")
print("Estimated time: 30-45 minutes")
print("Started at:", pd.Timestamp.now().strftime("%H:%M:%S"))

# Use your ultimate prompt
result_final = test_prompt(sentences, true_labels, prompt_ultimate,
                          "ULTIMATE PROMPT - ALL DATA", 
                          temperature=0.15)

print(f"\n{'='*60}")
print(f" FINAL RESULTS")
print(f"{'='*60}")
print(f"F1-Score:  {result_final['f1']:.3f}")
print(f"Accuracy:  {result_final['accuracy']:.3f}")
print(f"Completed: {pd.Timestamp.now().strftime('%H:%M:%S')}")

if result_final['f1'] >= 0.85:
    print("\n TARGET ACHIEVED!")
elif result_final['f1'] >= 0.75:
    print(f"\n Very close! Gap: {0.85 - result_final['f1']:.3f}")
else:
    print(f"\n Gap to target: {0.85 - result_final['f1']:.3f}")
    print("\nThis demonstrates comprehensive prompt engineering!")
    print("Further improvement likely requires fine-tuning.")



🎯 FINAL TEST: ALL DATA

Testing on 3722 sentences
Estimated time: 30-45 minutes
Started at: 18:56:07

Testing: ULTIMATE PROMPT - ALL DATA
Temperature: 0.15
Processed 25/3722
Processed 50/3722
Processed 75/3722
Processed 100/3722
Processed 125/3722
Processed 150/3722
Processed 175/3722
Processed 200/3722
Processed 225/3722
Processed 250/3722
Processed 275/3722
Processed 300/3722
Processed 325/3722
Processed 350/3722
Processed 375/3722
Processed 400/3722
Processed 425/3722
Processed 450/3722
Processed 475/3722
Processed 500/3722
Processed 525/3722
Processed 550/3722
Processed 575/3722
Processed 600/3722
Processed 625/3722
Processed 650/3722
Processed 675/3722
Processed 700/3722
Processed 725/3722
Processed 750/3722
Processed 775/3722
Processed 800/3722
Processed 825/3722
Processed 850/3722
Processed 875/3722
Processed 900/3722
Processed 925/3722
Processed 950/3722
Processed 975/3722
Processed 1000/3722
Processed 1025/3722
Processed 1050/3722
Processed 1075/3722
Processed 1100/3722
Proces